# CIFAR-100 Detection Temps Reel
**ResNet18 · 100 classes · Colab + Local**

## 0 · Environnement

In [4]:
import sys, os

# Détection automatique de l'environnement
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environnement :', 'Google Colab' if IN_COLAB else 'Local (PyCharm)')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device        :', DEVICE)
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
else:
    print('⚠️  Pas de GPU — CPU utilisé (plus lent)')


Environnement : Local (PyCharm)


ModuleNotFoundError: No module named 'torch'

## 1 · Installation

In [ ]:
# En local : pip install gradio torch torchvision pillow numpy
# En Colab  : exécuté automatiquement ci-dessous

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

required = ['gradio', 'torch', 'torchvision', 'pillow', 'numpy']
for pkg in required:
    try:
        __import__(pkg.split('[')[0])
        print(f'✅ {pkg} déjà installé')
    except ImportError:
        print(f'📦 Installation de {pkg}...')
        pip_install(pkg)
        print(f'✅ {pkg} installé')


✅ gradio déjà installé
✅ torch déjà installé
✅ torchvision déjà installé
📦 Installation de pillow...
✅ pillow installé
✅ numpy déjà installé


## 2 · Chemin du modele — MODIFIE ICI

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  MODIFIE ICI — chemin vers ton fichier best.pth ou best.keras
# ═══════════════════════════════════════════════════════════════

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Chemin Drive — modifie selon où tu as sauvegardé best.pth
    MODEL_PATH = '/content/drive/MyDrive/cifar100/checkpoints/best.pth'

else:
    # Chemin local Windows — modifie avec ton vrai chemin
    MODEL_PATH = r'C:\Users\ZIDAN\Desktop\Computer_V_project\notebooks\cv_model_efficientNet.keras'
    # Ou Linux/Mac :
    MODEL_PATH = r'C:\Users\ZIDAN\Desktop\Computer_V_project\notebooks\cv_model_efficientNet.keras'

# ─── Vérification ─────────────────────────────────────────────────────────────
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    ext     = os.path.splitext(MODEL_PATH)[1].lower()
    print(f'✅ Modèle trouvé : {MODEL_PATH}')
    print(f'   Taille  : {size_mb:.1f} MB')
    print(f'   Format  : {ext}')
    MODEL_FORMAT = ext  # '.pth' ou '.keras' ou '.h5'
else:
    print(f'❌ Fichier non trouvé : {MODEL_PATH}')
    print('   Vérifie le chemin ci-dessus')
    MODEL_FORMAT = None


✅ Modèle trouvé : C:\Users\ZIDAN\PycharmProjects\Computer_Vision_project\cifar100-project\checkpoints\best.pth
   Taille  : 89.8 MB
   Format  : .pth


## 3 · Classes CIFAR-100

In [ ]:
CIFAR100_CLASSES = [
    "apple","aquarium_fish","baby","bear","beaver","bed","bee","beetle",
    "bicycle","bottle","bowl","boy","bridge","bus","butterfly","camel",
    "can","castle","caterpillar","cattle","chair","chimpanzee","clock",
    "cloud","cockroach","couch","crab","crocodile","cup","dinosaur",
    "dolphin","elephant","flatfish","forest","fox","girl","hamster",
    "house","kangaroo","keyboard","lamp","lawn_mower","leopard","lion",
    "lizard","lobster","man","maple_tree","motorcycle","mountain",
    "mouse","mushroom","oak_tree","orange","orchid","otter","palm_tree",
    "pear","pickup_truck","pine_tree","plain","plate","poppy","porcupine",
    "possum","rabbit","raccoon","ray","road","rocket","rose","sea","seal",
    "shark","shrew","skunk","skyscraper","snail","snake","spider",
    "squirrel","streetcar","sunflower","sweet_pepper","table","tank",
    "telephone","television","tiger","tractor","train","trout","tulip",
    "turtle","wardrobe","whale","willow_tree","wolf","woman","worm",
]
print(f"{len(CIFAR100_CLASSES)} classes OK")


100 classes OK


## 4 · Modele ResNet18

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from PIL import Image

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))

class EfficientNet(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.in_ch  = 64
        self.conv1  = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(64)
        self.layer1 = self._make(64,  2, 1)
        self.layer2 = self._make(128, 2, 2)
        self.layer3 = self._make(256, 2, 2)
        self.layer4 = self._make(512, 2, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(512, num_classes)
    def _make(self, ch, n, stride):
        layers = [ResidualBlock(self.in_ch, ch, stride)]
        self.in_ch = ch
        for _ in range(1, n): layers.append(ResidualBlock(ch, ch))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        for layer in [self.layer1, self.layer2, self.layer3, self.layer4]: x = layer(x)
        return self.fc(torch.flatten(self.pool(x), 1))

model = framework = None

if MODEL_FORMAT in [".pth", ".pt"]:
    framework = "pytorch"
    model = EfficientNet(num_classes=100).to(DEVICE)
    ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
    state = ckpt.get("model", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    print("EfficientNet PyTorch charge OK")
elif MODEL_FORMAT in [".keras", ".h5"]:
    framework = "keras"
    import tensorflow as tf
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Keras charge OK — input:", model.input_shape)
else:
    framework = "demo"
    print("Mode DEMO actif")


ResNet18 PyTorch charge OK


## 5 · Inference

In [ ]:
import torchvision.transforms as T

preprocess = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

def predict(pil_img, top_k=5):
    img_rgb = pil_img.convert("RGB")
    if framework == "pytorch":
        x = preprocess(img_rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    elif framework == "keras":
        h, w  = model.input_shape[1], model.input_shape[2]
        arr   = np.array(img_rgb.resize((w, h))).astype("float32") # Pas de division par 255 pour EfficientNetV2
        probs = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    else:
        probs = np.random.dirichlet(np.ones(100))
    idx = probs.argsort()[::-1][:top_k]
    return [(CIFAR100_CLASSES[i], float(probs[i])) for i in idx]

r = predict(Image.new("RGB", (64, 64), (120, 80, 60)), top_k=3)
print("Test inference OK:", r)


Test inference OK: [('plain', 0.1862689107656479), ('sweet_pepper', 0.11388562619686127), ('sea', 0.05484367907047272)]


## 6 · Application Gradio — temps reel

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image, ImageDraw
import socket

# FIX 1: couleurs RGB sans canal alpha
def conf_color(s):
    return (34, 197, 94) if s > 0.6 else ((234, 179, 8) if s > 0.3 else (239, 68, 68))

def draw_overlay(pil_img, results):
    # FIX 2: convert("RGB") obligatoire — fill RGBA ne marche pas sur image RGB
    img  = pil_img.copy().convert("RGB").resize((480, 360))
    draw = ImageDraw.Draw(img)
    W, H = img.size
    pad  = 10
    oh   = min(len(results) * 38 + 20, 210)
    # FIX 3: fill=(R,G,B) sans 4eme canal
    draw.rectangle([0, H-oh, W, H], fill=(10, 10, 20))
    for i, (label, score) in enumerate(results):
        y     = H - oh + 10 + i * 38
        color = conf_color(score)
        bw    = max(0, int(score * (W - 2*pad - 130)))
        draw.rectangle([pad+120, y+6, W-pad,      y+28], fill=(40, 40, 55))
        draw.rectangle([pad+120, y+6, pad+120+bw, y+28], fill=color)
        draw.text((pad,    y+6), f"{i+1}. {label[:16]}", fill=(220, 220, 220))
        draw.text((W-58,   y+6), f"{score*100:5.1f}%",   fill=color)
    return img

def predict_stream(frame):
    if frame is None:
        return None, "En attente de la camera..."

    try:
        pil = Image.fromarray(frame) if isinstance(frame, np.ndarray) else frame
        results = predict(pil, top_k=5)

        if not results:
            return pil, "Aucun resultat."

        top = results[0]
        icon = "OK" if top[1] > 0.6 else ("~" if top[1] > 0.3 else "?")
        txt = (
            f"{icon} {top[0].upper()}  {top[1]*100:.1f}%\n\n"
            + "\n".join(
                f"  {i+1}. {l:<20} {s*100:.1f}%"
                for i, (l, s) in enumerate(results)
            )
        )

        return draw_overlay(pil, results), txt

    except Exception as e:
        return None, f"Erreur: {e}"



CSS = """
body, .gradio-container { background:#0a0a0f!important; font-family:Courier New,monospace!important; }
h1 { text-align:center; color:#38bdf8; font-size:1.5rem; letter-spacing:3px; padding:12px 0; }
.gradio-textbox textarea { background:#0f172a!important; color:#94a3b8!important; font-size:0.88rem!important; border:1px solid #1e293b!important; border-radius:8px!important; }
footer { display:none!important; }
"""

with gr.Blocks(title="CIFAR-100 Detector", css=CSS) as demo:
    gr.HTML("<h1>CIFAR-100 DETECTOR</h1>")
    gr.HTML("<p style=text-align:center;color:#475569;font-size:.82rem>EfficientNet · 100 classes · temps reel</p>")
    with gr.Row(equal_height=True):
        with gr.Column(scale=3):
            # FIX 4: webcam_options remplace mirror_webcam (supprime dans Gradio 4+)
            cam = gr.Image(
                sources=["webcam"],
                streaming=True,
                height=380,
                show_label=False,
                webcam_options=gr.WebcamOptions(mirror=False),
            )
        with gr.Column(scale=2):
            out_img  = gr.Image(height=260, show_label=False)
            out_text = gr.Textbox(lines=8, interactive=False, show_label=False,
                                  placeholder="Pointe ta camera vers un objet...")
    gr.HTML("<p style=text-align:center;color:#334155;font-size:.75rem>Sur telephone: ouvre le lien et autorise la camera</p>")
    # FIX 5: stream_every (remplace time_limit dans Gradio 4+)
    cam.stream(fn=predict_stream, inputs=[cam], outputs=[out_img, out_text], stream_every=0.1)

print()
print("="*55)
if IN_COLAB:
    print("Colab: lien public genere ci-dessous (72h)")
    print("Ouvre ce lien sur ton telephone")
    print("="*55)
    demo.launch(share=True, show_error=True)
else:
    try: ip = socket.gethostbyname(socket.gethostname())
    except: ip = "127.0.0.1"
    print(f"Local  -> http://localhost:7860")
    print(f"Telephone (meme Wi-Fi) -> http://{ip}:7860")
    print("Lien public: change share=False en share=True ci-dessous")
    print("="*55)
    demo.launch(server_name="0.0.0.0", server_port=7861, share=True, show_error=True)
# Pour lien public depuis PyCharm, remplace la ligne ci-dessus par:
# demo.launch(share=True, show_error=True)


C:\Users\ZIDAN\AppData\Local\Temp\ipykernel_6148\1696814507.py:55: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="CIFAR-100 Detector", css=CSS) as demo:



Local  -> http://localhost:7860
Telephone (meme Wi-Fi) -> http://192.168.1.188:7860
Lien public: change share=False en share=True ci-dessous
* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://5dffe8070b7593a0f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\ZIDAN\PyCharmMiscProject\TP_Env_Paquets\env_tp\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\ZIDAN\PyCharmMiscProject\TP_Env_Paquets\env_tp\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\ZIDAN\PyCharmMiscProject\TP_Env_Paquets\env_tp\Lib\site-packages\gradio\blocks.py", line 2192, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\ZIDAN\PyCharmMiscProject\TP_Env_Paquets\env_tp\Lib\site-packages\gradio\blocks.py", line 1886, in postpro

## 7 · (Optionnel) Test avec image uploadee

In [ ]:
import matplotlib.pyplot as plt

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    sources  = {n: __import__("io").BytesIO(d) for n, d in uploaded.items()}
else:
    sources = {"test.jpg": r"C:\Users\ZIDAN\Pictures\test.jpg"}

for name, src in sources.items():
    img     = Image.open(src).convert("RGB")
    results = predict(img, top_k=5)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.imshow(img); ax1.set_title(name); ax1.axis("off")
    labels = [r[0] for r in results[::-1]]
    scores = [r[1] for r in results[::-1]]
    colors = ["green" if s>.6 else ("orange" if s>.3 else "red") for s in scores]
    ax2.barh(labels, scores, color=colors)
    ax2.set_xlim(0, 1); ax2.set_title("Top-5 EfficientNet")
    for j, s in enumerate(scores):
        ax2.text(s+.01, j, f"{s*100:.1f}%", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"Top-1: {results[0][0]} ({results[0][1]*100:.1f}%)")
